# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print(metadata['name'] + ': ' + metadata['description'])
print('Published:', metadata.get('datePublished', 'N/A'))
print('Keywords:', ', '.join(metadata.get('keywords', [])))
print('License:', metadata.get('license'))

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Access record sets using their @id
record_sets_ids = []

# Croissant metadata exposes recordSet directly as a key
record_sets = dataset.metadata.to_json().get('recordSet', [])
if not record_sets:
    print('No recordSet entries found in the metadata - check the schema.')
else:
    for record_set in record_sets:
        if isinstance(record_set, dict):
            rid = record_set.get('@id')
        else:
            rid = record_set
        record_sets_ids.append(rid)
        print(f"RecordSet @id: {rid}")

# Preview fields for each record set
for rid in record_sets_ids:
    print(f"\nFields for RecordSet {rid}:")
    try:
        for x in dataset.records(record_set=rid):
            print(list(x.keys()))
            break  # preview only the first record
    except Exception as e:
        print('Error accessing records:', e)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
dataframes = {}

# If no recordSet IDs detected, try fallback to common pattern:
if not record_sets_ids:
    # Try default recordSet names based on Croissant conventions
    record_sets_ids = ['cr:recordSet']

for rs_id in record_sets_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded RecordSet {rs_id} with columns: {df.columns.tolist()}")
        print(df.head(2))
    except Exception as e:
        print(f"Could not load records for {rs_id}: {e}")

# Select a main record set for further exploration
main_record_set_id = None
for rs_id in record_sets_ids:
    if rs_id in dataframes and not dataframes[rs_id].empty:
        main_record_set_id = rs_id
        break
if main_record_set_id is None:
    raise Exception("No accessible record set found!")

print(f"\nUsing main record set: {main_record_set_id}")
print(f"Columns: {dataframes[main_record_set_id].columns.tolist()}")
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes for deeper analysis.

In [ ]:
# Choose a numeric field and a group field for demonstration
df = dataframes[main_record_set_id]

# Try GAD-7 score, PHQ-9 score, or PSQ score (column names may vary)
possible_numeric_fields = [col for col in df.columns if any(s in col.lower() for s in ['gad', 'phq', 'psq', 'score'])]
numeric_field = None
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]

if numeric_field is not None:
    print(f"Using numeric field: {numeric_field}")
    
    # Remove missing values
    num_series = pd.to_numeric(df[numeric_field], errors='coerce')
    clean_df = df[num_series.notnull()].copy()
    clean_df[numeric_field] = num_series[num_series.notnull()]

    # Example threshold
    threshold = clean_df[numeric_field].median()

    filtered_df = clean_df[clean_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping by categorical column
    group_field_candidates = [col for col in df.columns if col.lower() in ['gender', 'age', 'marital status', 'village', 'level_of_education']] + 
        [col for col in df.columns if any(c in col.lower() for c in ['gender', 'education', 'village', 'marital'])]
    group_field = None
    if group_field_candidates:
        group_field = group_field_candidates[0]
    
    if group_field is not None and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"Grouped data by {group_field} (mean of {numeric_field}):")
        print(grouped_df.head())
else:
    print('No suitable numeric field found for EDA in the record set.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field's distribution
if numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=20, kde=True, color='skyblue')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If group field available, visualize grouped means
    if group_field and group_field in df.columns:
        group_means = df.groupby(group_field)[numeric_field].mean().sort_values()
        plt.figure(figsize=(8, 4))
        sns.barplot(x=group_means.index, y=group_means.values)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

Through this notebook, we:
- Loaded the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using `mlcroissant`.
- Explored available record sets and fields using their `@id` identifiers.
- Extracted and previewed tabular data using pandas.
- Applied basic filtering and normalization to numeric mental health assessment scores.
- Grouped and visualized data to observe patterns across demographic variables.

Key observations:
- The dataset contains demographic and mental health indicator scores such as GAD-7, PHQ-9, and PSQ.
- Numeric scores vary across demographic groups, highlighting potential areas for further analysis.
- Visualization reveals distributional characteristics and enables comparison across key group fields.

For advanced analysis, consider extending with deeper data cleaning, statistical modeling, and richer visualizations as required for public health and research inquiries.